# ETL + PCA Solar

Pipeline de limpieza/estandarizacion y PCA en solar_efficiency.csv.

Conversion conceptual 1:1 desde el ejemplo R homologo.


In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "R").exists():
            return candidate
    raise FileNotFoundError("No se encontro la carpeta R del repositorio")

REPO_ROOT = find_repo_root(Path.cwd())
print("Repo root:", REPO_ROOT)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

solar_path = REPO_ROOT / "R" / "nuevos" / "4_fuentes_etl_pca" / "solar_efficiency.csv"
df = pd.read_csv(solar_path)

num_cols = [
    "ambient_temp", "module_temp", "irradiation",
    "temp_diff_module_ambient", "temp_excess", "efficiency_kwh_kwp"
]
X = df[num_cols].copy().fillna(df[num_cols].median(numeric_only=True))
X_scaled = StandardScaler().fit_transform(X)


In [ ]:
pca = PCA()
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
print("Varianza explicada acumulada:", explained.cumsum())

loadings = pd.DataFrame(
    pca.components_.T,
    index=num_cols,
    columns=[f"PC{i+1}" for i in range(len(num_cols))]
)
print(loadings.round(3))
